# Deep RL Results

Learning curves loaded from `results/` — **3 seeds each, mean ± 1 std**.

| Environment | Algorithms | Steps |
|---|---|---|
| CartPole-v1 | DQN vs PPO | 30 000 |
| Acrobot-v1 | DQN vs PPO | 50 000 |
| MountainCar-v0 | DQN | 40 000 |
| Pendulum-v1 | PPO vs SAC vs SAC_fixed_alpha vs TD3 | 30 000 |
| MountainCarContinuous-v0 | SAC vs SAC_fixed_alpha vs TD3 | 30 000 |

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils.plotting import (
    load_runs, aggregate, plot_learning_curves, plot_grid, summary_table,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1 · Discrete environments — DQN vs PPO

Both algorithms see the same step budget (30 k on CartPole, 50 k on Acrobot).  
DQN is off-policy (replay buffer); PPO is on-policy (rollouts).  
MountainCar is included as a documented hard-exploration failure case for DQN.

In [ ]:
fig = plot_grid([
    ("CartPole-v1",    ["DQN", "PPO"]),
    ("Acrobot-v1",     ["DQN", "PPO"]),
    ("MountainCar-v0", ["DQN"]),
], ncols=3)
fig.suptitle("Discrete environments", fontsize=14, y=1.02)
plt.show()

## 2 · CartPole-v1 — DQN vs PPO (detail)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_learning_curves(["DQN", "PPO"], "CartPole-v1", ax=ax)
ax.axhline(500, ls="--", color="gray", alpha=0.5, label="max score (500)")
ax.legend()
plt.show()

## 3 · Acrobot-v1 — DQN vs PPO (detail)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_learning_curves(["DQN", "PPO"], "Acrobot-v1", ax=ax)
ax.axhline(-100, ls="--", color="gray", alpha=0.5, label="solved ≈ −100")
ax.legend()
plt.show()

## 4 · Pendulum-v1 — PPO vs SAC vs SAC_fixed_alpha vs TD3

All four algorithms on the same 30 k step budget.  
Off-policy methods (SAC, TD3) converge much faster than on-policy PPO —  
the log-scale plot makes this sample-efficiency gap explicit.

In [ ]:
algos_pendulum = ["PPO", "SAC", "SAC_fixed_alpha", "TD3"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_learning_curves(algos_pendulum, "Pendulum-v1", ax=axes[0],
                     title="Pendulum-v1")
axes[0].axhline(-200, ls="--", color="gray", alpha=0.5, label="solved ≈ −200")
axes[0].legend()

plot_learning_curves(algos_pendulum, "Pendulum-v1", ax=axes[1],
                     title="Pendulum-v1 (log scale — sample efficiency)")
axes[1].set_xscale("log")
axes[1].axhline(-200, ls="--", color="gray", alpha=0.5, label="solved ≈ −200")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5 · MountainCarContinuous-v0 — SAC vs SAC_fixed_alpha vs TD3

All three algorithms trained for 30 k steps.  
Final eval ≈ 0 for all — documented 'don't-act' failure: with sparse +100 goal reward  
and −0.1·action² step penalty, zero action is locally optimal without exploration  
that discovers the goal first.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_learning_curves(["SAC", "SAC_fixed_alpha", "TD3"],
                     "MountainCarContinuous-v0", ax=ax)
plt.show()

## 6 · SAC: automatic α vs fixed α (α = 0.2)

Haarnoja et al. 2018a used a fixed entropy temperature; 2018b introduced  
automatic tuning via a learned α that satisfies a target entropy constraint.  
This compares both variants on both continuous environments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, env in zip(axes, ["Pendulum-v1", "MountainCarContinuous-v0"]):
    plot_learning_curves(["SAC", "SAC_fixed_alpha"], env, ax=ax, title=env)
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["SAC auto-α (2018b)", "SAC fixed-α=0.2 (2018a)"])

fig.suptitle("SAC: automatic α tuning vs fixed α", fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Summary table

Final and peak evaluation return (mean ± std across 3 seeds).

In [ ]:
df = summary_table([
    ("DQN",             "CartPole-v1"),
    ("PPO",             "CartPole-v1"),
    ("DQN",             "Acrobot-v1"),
    ("PPO",             "Acrobot-v1"),
    ("DQN",             "MountainCar-v0"),
    ("PPO",             "Pendulum-v1"),
    ("SAC",             "Pendulum-v1"),
    ("SAC_fixed_alpha", "Pendulum-v1"),
    ("TD3",             "Pendulum-v1"),
    ("SAC",             "MountainCarContinuous-v0"),
    ("SAC_fixed_alpha", "MountainCarContinuous-v0"),
    ("TD3",             "MountainCarContinuous-v0"),
])
df.round(1)